# NB_patient_360
Builds **`<catalog>.gold.patient_360`** from the silver layer, as defined in the `sttm_silver_to_gold.xlsx` STTM (sheet `patient_360`).

* **Grain:** one row per patient
* **Driving table:** `silver.patients`
* **Joins (LEFT):** `silver.hospitals` on `registered_hospital_id = hospital_id`; `silver.insurance_providers` on `insurance_provider_id`; `silver.lab_results` pre-aggregated by `patient_id`
* **Filters:** none (all patients included)
* **Load type:** FULL, `CREATE OR REPLACE TABLE ... AS SELECT`

Note: `silver.patients.date_of_birth` is a TIMESTAMP, so it is cast to DATE to match the STTM target type.

In [0]:
dbutils.widgets.text("catalog", "adb_caresync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

source_schema = f"{catalog}.silver"
target_table = f"{catalog}.gold.patient_360"

print("=" * 80)
print("STEP 1: Input parameters")
print("=" * 80)
print(f"  catalog       : '{catalog}'")
print(f"  Source schema : {source_schema}")
print(f"  Target table  : {target_table}")

In [0]:
patient_360_sql = f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH lab_agg AS (
  -- Lab results aggregated to patient grain
  SELECT
    patient_id,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count,
    MAX(test_date)                        AS last_test_date
  FROM {source_schema}.lab_results
  GROUP BY patient_id
)
SELECT
  CAST(p.patient_id AS INT)                                                        AS patient_id,
  p.first_name,
  p.last_name,
  CAST(p.date_of_birth AS DATE)                                                    AS date_of_birth,
  CAST(FLOOR(DATEDIFF(current_date(), CAST(p.date_of_birth AS DATE)) / 365.25) AS INT) AS age,
  p.gender,
  p.city,
  p.state,
  CAST(p.registered_hospital_id AS INT)                                            AS hospital_id,
  h.hospital_name,
  CAST(p.insurance_provider_id AS INT)                                             AS insurance_provider_id,
  i.provider_name                                                                  AS insurance_provider_name,
  CAST(COALESCE(l.total_tests, 0) AS INT)                                          AS total_tests,
  CAST(COALESCE(l.abnormal_test_count, 0) AS INT)                                  AS abnormal_test_count,
  CAST(COALESCE(l.critical_test_count, 0) AS INT)                                  AS critical_test_count,
  l.last_test_date
FROM {source_schema}.patients p
LEFT JOIN {source_schema}.hospitals h
  ON p.registered_hospital_id = h.hospital_id
LEFT JOIN {source_schema}.insurance_providers i
  ON p.insurance_provider_id = i.insurance_provider_id
LEFT JOIN lab_agg l
  ON p.patient_id = l.patient_id
"""

print("=" * 80)
print("STEP 2: Building gold table (FULL load)")
print("=" * 80)
print("Executing query:")
print(patient_360_sql)

spark.sql(patient_360_sql)
print(f"FULL load completed -> {target_table}")

In [0]:
print("=" * 80)
print("STEP 3: Validation")
print("=" * 80)

source_count = spark.table(f"{source_schema}.patients").count()
target_df = spark.table(target_table)
target_count = target_df.count()
distinct_keys = target_df.select("patient_id").distinct().count()

print(f"  Source patients        : {source_count}")
print(f"  Target rows            : {target_count}")
print(f"  Distinct patient_id    : {distinct_keys}")

if target_count != source_count:
    raise ValueError("Row count mismatch: LEFT JOIN policy requires one row per source patient")
if distinct_keys != target_count:
    raise ValueError("Grain violation: patient_id is not unique in target")
print("Validation passed.")

#display(target_df.limit(10))

In [0]:
record_count = target_df.count()
print(f"Records in {target_table}: {record_count}")
dbutils.notebook.exit(str(record_count))